In [1]:
#Apriori algorithm
import pandas as pd
import numpy as np
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

In [2]:
dataset = [
    ['Coffee', 'Donut', 'Sandwich'],
    ['Coffee', 'Donut'],
    ['Coffee', 'Sandwich'],
    ['Coffee', 'Muffin'],
    ['Donut', 'Muffin']
]


In [3]:
# Q1. What does the dataset represent? List all transactions
for t in dataset:
    print("Transaction:", t)

Transaction: ['Coffee', 'Donut', 'Sandwich']
Transaction: ['Coffee', 'Donut']
Transaction: ['Coffee', 'Sandwich']
Transaction: ['Coffee', 'Muffin']
Transaction: ['Donut', 'Muffin']


In [4]:
# Q2. One-hot encode to DataFrame
te = TransactionEncoder()
te_ary = te.fit(dataset).transform(dataset)
df_ohe = pd.DataFrame(te_ary, columns=te.columns_)
print(df_ohe)
print("-" * 100)
print("Explanation: each column is an item (e.g., 'Coffee'), each row is a transaction; True means item present.")
print("-" * 100)

   Coffee  Donut  Muffin  Sandwich
0    True   True   False      True
1    True   True   False     False
2    True  False   False      True
3    True  False    True     False
4   False   True    True     False
----------------------------------------------------------------------------------------------------
Explanation: each column is an item (e.g., 'Coffee'), each row is a transaction; True means item present.
----------------------------------------------------------------------------------------------------


In [5]:
# Q3. Frequent itemsets with min_support = 0.4
min_support = 0.4
frequent_itemsets = apriori(df_ohe, min_support=min_support, use_colnames=True)

In [6]:
print(frequent_itemsets.sort_values(by=['support','itemsets'], ascending=[False, True]))

   support            itemsets
0      0.8            (Coffee)
1      0.6             (Donut)
2      0.4            (Muffin)
3      0.4          (Sandwich)
4      0.4     (Coffee, Donut)
5      0.4  (Coffee, Sandwich)


In [7]:
# Q4. Generate all association rules
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0)
print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']])

  antecedents consequents  support  confidence      lift
0    (Coffee)     (Donut)      0.4    0.500000  0.833333
1     (Donut)    (Coffee)      0.4    0.666667  0.833333
2    (Coffee)  (Sandwich)      0.4    0.500000  1.250000
3  (Sandwich)    (Coffee)      0.4    1.000000  1.250000


In [8]:
# Q5. Which rules satisfy min_support >= 0.4 and min_confidence >= 0.6?
strong_rules = rules[(rules['support'] >= 0.4) & (rules['confidence'] >= 0.6)]
print(strong_rules[['antecedents','consequents','support','confidence','lift']])

  antecedents consequents  support  confidence      lift
1     (Donut)    (Coffee)      0.4    0.666667  0.833333
3  (Sandwich)    (Coffee)      0.4    1.000000  1.250000


In [9]:
# Q6. Interpret one strong rule in words (if any)
if not strong_rules.empty:
    r = strong_rules.iloc[0]
    ant = ', '.join(list(r['antecedents']))
    con = ', '.join(list(r['consequents']))
    print(f"If a customer buys [{ant}], they are likely to buy [{con}] "
          f"(confidence={r['confidence']:.2f}, lift={r['lift']:.2f}).")
else:
    print("No strong rules found for given thresholds.")


If a customer buys [Donut], they are likely to buy [Coffee] (confidence=0.67, lift=0.83).


In [12]:
# Q7. Experiment: effect of changing min_support and min_confidence

for ms in [0.2, 0.4, 0.6]:
    for mc in [0.5, 0.7]:
        fi = apriori(df_ohe, min_support=ms, use_colnames=True)
        rls = association_rules(fi, metric="confidence", min_threshold=mc)
        print(f"min_support={ms}, min_confidence={mc} -> "
              f"itemsets={len(fi)}, rules={len(rls)}")

min_support=0.2, min_confidence=0.5 -> itemsets=10, rules=11
min_support=0.2, min_confidence=0.7 -> itemsets=10, rules=2
min_support=0.4, min_confidence=0.5 -> itemsets=6, rules=4
min_support=0.4, min_confidence=0.7 -> itemsets=6, rules=1
min_support=0.6, min_confidence=0.5 -> itemsets=2, rules=0
min_support=0.6, min_confidence=0.7 -> itemsets=2, rules=0


In [14]:
# Q8. Explain why Lift > 1 indicates a good association rule.

r = rules[rules['lift'] > 1].sort_values('lift', ascending=False).head(1)

if not r.empty:
    ant, con, lift = list(r.iloc[0]['antecedents']), list(r.iloc[0]['consequents']), r.iloc[0]['lift']
    print(f"If a customer buys {ant}, they are likely to buy {con} (Lift={lift:.2f}).")
    print("Explanation: Lift > 1 means buying the antecedent increases the chance of buying the consequent,")
    print("so the items are positively associated (a good rule).")
else:
    print("No rule with lift > 1 found in this dataset.")


If a customer buys ['Coffee'], they are likely to buy ['Sandwich'] (Lift=1.25).
Explanation: Lift > 1 means buying the antecedent increases the chance of buying the consequent,
so the items are positively associated (a good rule).
